In [133]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
from pathlib import Path
from brainvision.constants import *
from brainvision.utils import build_run_name, seed_everything
from brainvision.device import get_device, print_device_info

seed_everything(SPLIT_SEED)


In [134]:
# ══════════════════════════════════════════════════════════════════════════════
#                        CONFIGURE EXPERIMENT HERE
# ══════════════════════════════════════════════════════════════════════════════

MODEL    = "2D-CNN-Fabelo"      # "1D-NN" | "1D-NN-Fabelo" | "1D-CNN" | "2D-CNN" | "3D-CNN" | "SF"
LOSS_FN  = "UFL"                 # "CE" | "FL" | "DL" | "UFL"
REDUCE_PIXELS = True            # apply K-Means reduction to training set

STRATEGY = "vp_fabelo"          # "vp1" | "vp2" | "vp3" | "lopo" | "vp_fabelo"
N_FOLDS  = 5                    # used for vp3 and lopo only; vp_fabelo always uses 5 folds
TARGET_FOLDS = -1

In [135]:
from brainvision.data import load_all_campaigns

campaigns = load_all_campaigns(processed_dirs=PROCESSED_DIRS)

Campaigns loaded from memory cache (61 patients across 3 campaigns)


In [136]:
from brainvision.validation import get_splits

all_splits = get_splits(campaigns, STRATEGY)

In [137]:
from brainvision.losses import DiceLoss, FocalLoss, UnifiedFocalLoss
from brainvision.models import Baseline1DDNN, FabeloDNN, HuEtAl1DCNN, LeeEtAl2DCNN, HamidaEtAl3DCNN, HybridSN, SpectralFormer, Fabelo2DCNN, Simple2DCNN

MODEL_REGISTRY = {
    '1D-NN': lambda: Baseline1DDNN(
        input_channels  = N_DECIMATED_BANDS,
        n_classes       = N_CLASSES,
        dropout         = True,
        dropout_rate    = DROPOUT_RATE
    ),

    '1D-NN-Fabelo': lambda: FabeloDNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
    ),

    '1D-CNN': lambda: HuEtAl1DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES
    ),

    '2D-CNN': lambda: LeeEtAl2DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    '2D-CNN-Fabelo': lambda: Fabelo2DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),

    '2D-CNN-Simple': lambda: Simple2DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE,
    ),

    '3D-CNN': lambda: HamidaEtAl3DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    'HybridSN': lambda: HybridSN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    'SpectralFormer': lambda: SpectralFormer(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        near_band      = SF_NEAR_BAND,
        dim            = SF_DIM,
        depth          = SF_DEPTH,
        heads          = SF_HEADS,
        dim_head       = SF_DIM_HEAD,
        mlp_dim        = SF_MLP_DIM,
        dropout        = SF_DROPOUT,
        emb_dropout    = SF_EMB_DROPOUT,
        mode           = SF_MODE,
    ),
}

PATCH_MODELS = {'2D-CNN', '3D-CNN', 'HybridSN', '2D-CNN-Fabelo', '2D-CNN-Simple'}

LOSS_REGISTRY = {
    'CE'  : lambda w: nn.CrossEntropyLoss(weight=w),
    'FL'  : lambda w: FocalLoss(alpha=w),
    'DL'  : lambda w: DiceLoss(),
    'UFL' : lambda w: UnifiedFocalLoss(alpha=w),
}

In [138]:
device = get_device()
print_device_info()

  Device     : MPS
  Name       : Apple Silicon (MPS)
  Memory     : 19,070 MB total  | N/A free  | N/A used


In [139]:
def compute_class_weights(dataset: Dataset) -> torch.Tensor:
    counts  = torch.bincount(dataset.y, minlength=N_CLASSES).float()
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum()
    print("\nClass weights (from training set):")
    for i, (c, w) in enumerate(zip(counts, weights)):
        print(f"  {CLASS_NAMES[i+1]:<25}: {int(c):>10,} px  →  {w:.4f}")
    return weights

In [140]:
from brainvision.data import HSIPatchDataset, HSIPixelDataset, reduce_training_pixels
from brainvision.device import supports_pin_memory


def build_loaders(train_patients, val_patients,
                  reduce_pixels:  bool = REDUCE_PIXELS,
                  n_per_class:    int  = N_PIXELS_PER_CLASS):
    pin = supports_pin_memory(device)

    if MODEL in PATCH_MODELS:
        tr_ds = HSIPatchDataset(
            train_patients,
            patch_size = PATCH_SIZE,
            balance    = reduce_pixels,
            augment    = True,
        )
        va_ds = HSIPatchDataset(
            val_patients,
            patch_size = PATCH_SIZE,
            balance    = False,
            augment    = False,
        )

    else:
        tr_ds = HSIPixelDataset(train_patients)
        va_ds = HSIPixelDataset(val_patients)

        if reduce_pixels:
            tr_ds = reduce_training_pixels(tr_ds, n_per_class=n_per_class)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=pin)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=pin)
    weights   = compute_class_weights(tr_ds)

    print(f"\n  Train      : {len(tr_ds):,} samples")
    print(f"  Val        : {len(va_ds):,} samples")
    print(f"  Pin memory : {pin}")

    return tr_loader, va_loader, weights

## Training

In [141]:
def train_one_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * len(y)
        n          += len(y)
    return total_loss / n

In [142]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, n          = 0.0, 0
    all_preds, all_targets = [], []

    for X, y in loader:
        X, y   = X.to(device), y.to(device)
        logits = model(X)
        total_loss  += criterion(logits, y).item() * len(y)
        n           += len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    preds   = np.array(all_preds)
    targets = np.array(all_targets)

    macro_f1        = f1_score(targets, preds, average='macro', zero_division=0)
    macro_f1_no_bg  = f1_score(targets, preds, labels=[0, 1, 2],
                            average='macro', zero_division=0)
    sensitivity, specificity = _per_class_sens_spec(targets, preds)

    dice = np.zeros(N_CLASSES)
    for i in range(N_CLASSES):
        TP      = ((preds == i) & (targets == i)).sum()
        FP      = ((preds == i) & (targets != i)).sum()
        FN      = ((preds != i) & (targets == i)).sum()
        dice[i] = (2 * TP) / (2 * TP + FP + FN + 1e-6)

    macro_dice       = dice.mean()
    macro_dice_no_bg = dice[:3].mean()    # NT, TT, BV only

    return (total_loss / n,
            macro_f1,
            macro_f1_no_bg,
            sensitivity.mean(),
            specificity.mean(),
            macro_dice,
            macro_dice_no_bg)


def _per_class_sens_spec(targets: np.ndarray,
                          preds:   np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute per-class sensitivity (recall) and specificity from
    a confusion matrix. Returns arrays of shape (N_CLASSES,).
    """
    sensitivity = np.zeros(N_CLASSES)
    specificity = np.zeros(N_CLASSES)

    for i in range(N_CLASSES):
        TP = ((preds == i) & (targets == i)).sum()
        FN = ((preds != i) & (targets == i)).sum()
        FP = ((preds == i) & (targets != i)).sum()
        TN = ((preds != i) & (targets != i)).sum()

        sensitivity[i] = TP / (TP + FN + 1e-6)
        specificity[i] = TN / (TN + FP + 1e-6)

    return sensitivity, specificity

In [143]:
def train(model, train_loader, val_loader, criterion, device,
          checkpoint_path: str) -> dict:

    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimiser, mode='max',
                                  factor=LR_DECAY_FACTOR,
                                  patience=LR_DECAY_PATIENCE)

    history: dict = {
        'train_loss'       : [],
        'val_loss'         : [],
        'val_f1'           : [],
        'val_f1_no_bg'     : [],
        'val_sens'         : [],
        'val_spec'         : [],
        'val_dice'         : [],
        'val_dice_no_bg'   : [],
    }
    best_f1         = -1.0
    best_epoch      = 0
    no_improve      = 0
    prev_train_loss = float('inf')

    print(f"\n  {'Epoch':>5}  {'Train Loss':>11}  {'Val Loss':>9}  "
          f"{'Val F1':>8}  {'F1 -BG':>8}  {'Dice -BG':>9}  "
          f"{'Sens':>7}  {'Spec':>7}  {'LR':>9}  {'Time':>6}")
    print(f"  {'─'*95}")

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()

        train_loss = train_one_epoch(model, train_loader,
                                     optimiser, criterion, device)

        (val_loss, val_f1, val_f1_no_bg,
         val_sens, val_spec,
         val_dice, val_dice_no_bg) = evaluate(
            model, val_loader, criterion, device
        )

        scheduler.step(val_f1_no_bg)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_f1_no_bg'].append(val_f1_no_bg)
        history['val_sens'].append(val_sens)
        history['val_spec'].append(val_spec)
        history['val_dice'].append(val_dice)
        history['val_dice_no_bg'].append(val_dice_no_bg)

        lr     = optimiser.param_groups[0]['lr']
        marker = " ★" if val_f1_no_bg > best_f1 else ""

        print(f"  {epoch:>5}  {train_loss:>11.4f}  {val_loss:>9.4f}  "
              f"{val_f1:>8.4f}  {val_f1_no_bg:>8.4f}  "
              f"{val_dice_no_bg:>9.4f}  "
              f"{val_sens:>7.4f}  {val_spec:>7.4f}  "
              f"{lr:>9.2e}  {time.time()-t0:>5.1f}s{marker}")

        if val_f1_no_bg > best_f1:
            best_f1    = val_f1_no_bg
            best_epoch = epoch
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            no_improve += 1

        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"\n  ⏹  Early stopping at epoch {epoch} "
                  f"— val F1-noBG did not improve for "
                  f"{EARLY_STOP_PATIENCE} epochs")
            break

        train_loss_delta = abs(prev_train_loss - train_loss)
        if (epoch > EARLY_STOP_MIN_EPOCHS and
                train_loss_delta < TRAIN_LOSS_DELTA_MIN and
                no_improve > 0):
            print(f"\n  ⏹  Early stopping at epoch {epoch} "
                  f"— train loss plateaued (Δ={train_loss_delta:.6f}) "
                  f"and val F1-noBG not improving")
            break

        prev_train_loss = train_loss

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))

    history['best_epoch']        = best_epoch
    history['best_f1']           = best_f1
    history['best_f1_all']       = history['val_f1'][best_epoch - 1]
    history['best_sens']         = history['val_sens'][best_epoch - 1]
    history['best_spec']         = history['val_spec'][best_epoch - 1]
    history['best_dice']         = history['val_dice'][best_epoch - 1]
    history['best_dice_no_bg']   = history['val_dice_no_bg'][best_epoch - 1]

    print(f"\n  ✅ Best val macro-F1 (no BG) : {best_f1:.4f}  "
          f"Dice (no BG): {history['best_dice_no_bg']:.4f}  "
          f"F1 (all): {history['best_f1_all']:.4f}  "
          f"Sens: {history['best_sens']:.4f}  "
          f"Spec: {history['best_spec']:.4f}  "
          f"@ epoch {best_epoch}")

    return history

In [144]:
assert MODEL   in MODEL_REGISTRY, \
    f"Unknown MODEL '{MODEL}'. Choose from: {list(MODEL_REGISTRY.keys())}"
assert LOSS_FN in LOSS_REGISTRY,  \
    f"Unknown LOSS_FN '{LOSS_FN}'. Choose from: {list(LOSS_REGISTRY.keys())}"

Path(CHECKPOINTS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print(f"Device   : {device}")
print(f"Model    : {MODEL}")
print(f"Loss     : {LOSS_FN}")
print(f"Strategy : {STRATEGY}")

whitelist_folds = set()

if TARGET_FOLDS is not None:
    if isinstance(TARGET_FOLDS, int) and TARGET_FOLDS != -1:
        whitelist_folds.add(TARGET_FOLDS)
    elif isinstance(TARGET_FOLDS, (list, tuple)):
        whitelist_folds.update(TARGET_FOLDS)
    

all_histories = []

for i, split in enumerate(all_splits):
    if whitelist_folds and (i + 1) not in whitelist_folds:
        print(f"⏭️ Skipping fold {i + 1}...")
        continue
    
    fold_label = f"fold{split['fold']}" if split['fold'] else STRATEGY

    run_name = build_run_name(
        model         = MODEL,
        loss_fn       = LOSS_FN,
        strategy      = STRATEGY,
        fold          = split['fold'],
        reduce_pixels = REDUCE_PIXELS,
    )

    print(f"\n{'═'*60}")
    print(f"  Run: {run_name}")
    print(f"  Train: {len(split['train'])} images  "
          f"Val: {len(split['val'])} images  "
          f"Test: {len(split['test'])} images")
    print(f"{'═'*60}")

    checkpoint_path = f"{CHECKPOINTS_DIR}/{run_name}.pt"

    train_loader, val_loader, class_weights = build_loaders(
        split['train'], split['val'],
        reduce_pixels  = REDUCE_PIXELS,
        n_per_class    = N_PIXELS_PER_CLASS,
    )

    model     = MODEL_REGISTRY[MODEL]().to(device)
    criterion = LOSS_REGISTRY[LOSS_FN](class_weights.to(device))

    print(f"\n  Params : {sum(p.numel() for p in model.parameters()):,}")

    history = train(
        model           = model,
        train_loader    = train_loader,
        val_loader      = val_loader,
        criterion       = criterion,
        device          = device,
        checkpoint_path = checkpoint_path,
    )

    history['run_name']      = run_name
    history['strategy']      = STRATEGY        # ← add
    history['balance']       = REDUCE_PIXELS   # ← add
    history['test_patients'] = [p['id'] for p in split['test']]
    all_histories.append(history)

    np.save(f"{RESULTS_DIR}/{run_name}_history.npy", history)
    print(f"  History   → {RESULTS_DIR}/{run_name}_history.npy")
    print(f"  Checkpoint → {checkpoint_path}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  {'Run':<40} {'Best F1':>8}  {'Epoch':>6}")
print(f"  {'─'*55}")
for h in all_histories:
    print(f"  {h['run_name']:<40} {h['best_f1']:>8.4f}  {h['best_epoch']:>6}")

Device   : mps
Model    : 2D-CNN-Fabelo
Loss     : UFL
Strategy : vp_fabelo

════════════════════════════════════════════════════════════
  Run: 2dcnnfabelo_ufl_bal_fold1_vpfabelo
  Train: 34 images  Val: 12 images  Test: 15 images
════════════════════════════════════════════════════════════
  Balancing → 19739 centres per class
  Lazy dataset: 78,956 patches (extracted on-the-fly)
  Lazy dataset: 128,322 patches (extracted on-the-fly)

Class weights (from training set):
  Normal Tissue (NT)       :     19,739 px  →  0.2500
  Tumour Tissue (TT)       :     19,739 px  →  0.2500
  Blood Vessel (BV)        :     19,739 px  →  0.2500
  Background (BG)          :     19,739 px  →  0.2500

  Train      : 78,956 samples
  Val        : 128,322 samples
  Pin memory : False

  Params : 142,052

  Epoch   Train Loss   Val Loss    Val F1    F1 -BG   Dice -BG     Sens     Spec         LR    Time
  ───────────────────────────────────────────────────────────────────────────────────────────────
      